# Notebook 4 — Simulation Study, Scenario 1 (lambda=0)


In [ ]:
# شماره بخش اجرا
PART = 4        # دفعه اول
# PART = 2      # دفعه دوم
# ...
# PART = 10     # دفعه دهم

In [ ]:

# =====================================================
# Imports
# =====================================================
import os, gc, json, time, datetime, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from scipy.special import kv, gamma
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.linalg import cholesky
from scipy.optimize import curve_fit

import pymc as pm
import arviz as az

np.set_printoptions(precision=4, suppress=True, threshold=20)
print('Python:', platform.python_version())
print('PyMC:', pm.__version__)
print('ArviZ:', az.__version__)


In [ ]:

# =====================================================
# Global configuration
# =====================================================
CONFIG = {
    # Change to 100 for the final table in the paper.
    'N_REPS': 50,
    'base_seed': 2030,

    # Grid and low-rank structure
    'n_x': 15, 'n_y': 15, 'n_t': 8,
    'xmin': 0.0, 'xmax': 15.0, 'ymin': 0.0, 'ymax': 15.0,
    'K': 20, 'J': 9,
    'rbf_bandwidth': None,
    'jitter': 1e-6,

    # True parameters, scenario 1: no skewness
    'beta0': 1.0, 'beta1': 2.0, 'beta2': -1.0,
    'sigma': 1.0,
    'sigma_theta': 1.0,
    'phi_s': 4.0, 'phi_t': 2.0,
    'nu_s': 1.5, 'nu_t': 1.0,
    'lambda': 0.0,

    # Identifiability controls copied from the final Notebook 1/2/3 logic
    'restrict_B_against_X': True,
    'normalize_B_row_rms': True,
    'min_basis_scale': 1e-12,
    'fix_sigma_theta': True,

    # Variogram fit settings copied from final Notebook 2/3 logic
    'max_pairs_space': 120_000,
    'max_pairs_time': 120_000,
    'n_bins_space': 12,
    'n_bins_time': 7,
    'min_pairs_per_bin': 20,
    'phi_s_bounds': (1.0, 10.0),
    'phi_t_bounds': (0.5, 4.0),
    'nu_s_bounds': (0.5, 3.0),
    'nu_t_bounds': (0.5, 2.5),
    'variogram_prior_weight': 0.15,
    'prior_phi_s': 4.0,
    'prior_phi_t': 2.0,
    'prior_nu_s': 1.5,
    'prior_nu_t': 1.0,

    # PyMC settings: increase for final paper if compute allows
    'nuts_draws': 600,
    'nuts_tune': 600,
    'nuts_chains': 2,
    'nuts_cores': 2,   # use 2 if Kaggle resources allow
    'nuts_target_accept': 0.93,
    'nuts_max_treedepth': 9,

    'metro_draws': 600,
    'metro_tune': 600,
    'metro_chains': 2,
    'metro_cores': 2,  # use 2 if Kaggle resources allow

    # Priors
    'beta_sd': 10.0,
    'sigma_halfnormal_sd': 2.0,
    'lambda_sd': 1.5,
    'lambda_lower': -5.0,
    'lambda_upper': 5.0,

    # Output
    'results_dir': '/kaggle/working/Results_Simulation100_Lambda0_AHMC_MCMC'
}
if not os.path.exists('/kaggle/working'):
    CONFIG['results_dir'] = '/mnt/data/Results_Simulation100_Lambda0_AHMC_MCMC'
os.makedirs(CONFIG['results_dir'], exist_ok=True)
print('Results directory:', CONFIG['results_dir'])
print('N =', CONFIG['n_x']*CONFIG['n_y']*CONFIG['n_t'], ' r =', CONFIG['K']*CONFIG['J'])


In [ ]:

# =====================================================
# Data-generation functions from finalized Notebook 1
# =====================================================
def generate_spatial_grid(cfg):
    xs = np.linspace(cfg['xmin'], cfg['xmax'], cfg['n_x'])
    ys = np.linspace(cfg['ymin'], cfg['ymax'], cfg['n_y'])
    return np.array([(x, y) for x in xs for y in ys], dtype=np.float64)

def generate_time_grid(cfg):
    return np.arange(1, cfg['n_t'] + 1, dtype=np.float64)

def build_st_locations(cfg):
    S = generate_spatial_grid(cfg)
    T = generate_time_grid(cfg)
    ST = np.array([[s[0], s[1], t] for s in S for t in T], dtype=np.float64)
    return ST, S, T

def design_matrix(ST, cfg):
    x_scaled = ST[:, 0] / cfg['xmax']
    y_scaled = ST[:, 1] / cfg['ymax']
    return np.column_stack([np.ones(ST.shape[0]), x_scaled, y_scaled]).astype(np.float64)

def matern_corr(D, phi, nu):
    D = np.asarray(D, dtype=np.float64)
    R = np.ones_like(D, dtype=np.float64)
    idx = D > 0
    z = np.sqrt(2.0 * nu) * D[idx] / max(phi, 1e-8)
    R[idx] = (1.0 / (gamma(nu) * 2.0 ** (nu - 1.0))) * (z ** nu) * kv(nu, z)
    R = np.nan_to_num(R, nan=1.0, posinf=0.0, neginf=0.0)
    return np.clip(R, 0.0, 1.0)

def choose_spatial_centers(S, K, rng):
    idx = rng.choice(S.shape[0], size=K, replace=False)
    return S[idx].copy(), idx

def compute_default_bandwidth(S, centers):
    D = cdist(S, centers)
    nearest = np.min(D, axis=1)
    bw = np.median(nearest[nearest > 0])
    if not np.isfinite(bw) or bw <= 0:
        DD = cdist(centers, centers)
        bw = np.median(DD[DD > 0])
    return float(max(bw, 1e-3))

def spatial_rbf_basis(spatial_locations, centers, bandwidth):
    D2 = cdist(spatial_locations, centers) ** 2
    B = np.exp(-D2 / (2.0 * bandwidth ** 2))
    B = B / np.maximum(B.sum(axis=1, keepdims=True), 1e-12)
    return B.astype(np.float64)

def temporal_fourier_basis(times, J):
    times = np.asarray(times, dtype=np.float64)
    t_scaled = (times - times.min()) / (times.max() - times.min() + 1e-12)
    cols = [np.ones_like(t_scaled)]
    for m in range(1, (J - 1) // 2 + 1):
        cols.append(np.sin(2 * np.pi * m * t_scaled))
        cols.append(np.cos(2 * np.pi * m * t_scaled))
    B = np.column_stack(cols)
    if B.shape[1] < J:
        B = np.column_stack([B, t_scaled])
    return B[:, :J].astype(np.float64)

def build_low_rank_basis(ST, S, T, B_s_unique, B_t_unique):
    n_s = S.shape[0]
    n_t = T.shape[0]
    K = B_s_unique.shape[1]
    J = B_t_unique.shape[1]
    B = np.zeros((n_s * n_t, K * J), dtype=np.float64)
    row = 0
    for si in range(n_s):
        for tj in range(n_t):
            B[row, :] = np.kron(B_t_unique[tj, :], B_s_unique[si, :])
            row += 1
    return B

def restrict_and_normalize_B(B_raw, X, cfg):
    B = np.asarray(B_raw, dtype=np.float64).copy()
    if cfg.get('restrict_B_against_X', True):
        Q, _ = np.linalg.qr(X, mode='reduced')
        B = B - Q @ (Q.T @ B)
    basis_scale = 1.0
    if cfg.get('normalize_B_row_rms', True):
        row_rms = np.sqrt(np.mean(np.sum(B ** 2, axis=1)))
        basis_scale = max(float(row_rms), cfg.get('min_basis_scale', 1e-12))
        B = B / basis_scale
    return B.astype(np.float64), basis_scale

def coefficient_covariance(centers_s, J, cfg):
    K = centers_s.shape[0]
    Ds = cdist(centers_s, centers_s)
    Cs = matern_corr(Ds, cfg['phi_s'], cfg['nu_s']) + cfg['jitter'] * np.eye(K)
    tau = np.arange(1, J + 1, dtype=np.float64).reshape(-1, 1)
    Dt = cdist(tau, tau)
    Ct = matern_corr(Dt, cfg['phi_t'], cfg['nu_t']) + cfg['jitter'] * np.eye(J)
    C_theta = np.kron(Ct, Cs) + cfg['jitter'] * np.eye(K * J)
    return C_theta, Cs, Ct

def simulate_lowrank_dataset(cfg, seed):
    rng = np.random.default_rng(seed)
    ST, S, T = build_st_locations(cfg)
    X = design_matrix(ST, cfg)
    centers_s, center_indices = choose_spatial_centers(S, cfg['K'], rng)
    ell_s = compute_default_bandwidth(S, centers_s) if cfg['rbf_bandwidth'] is None else float(cfg['rbf_bandwidth'])
    B_s_unique = spatial_rbf_basis(S, centers_s, ell_s)
    B_t_unique = temporal_fourier_basis(T, cfg['J'])
    B_raw = build_low_rank_basis(ST, S, T, B_s_unique, B_t_unique)
    B, basis_scale = restrict_and_normalize_B(B_raw, X, cfg)
    C_theta, Cs_centers, Ct_basis = coefficient_covariance(centers_s, cfg['J'], cfg)
    L_theta = cholesky(C_theta, lower=True, check_finite=False)
    r = cfg['K'] * cfg['J']
    theta = cfg['sigma_theta'] * (L_theta @ rng.normal(size=r))
    beta = np.array([cfg['beta0'], cfg['beta1'], cfg['beta2']], dtype=np.float64)
    mu = X @ beta + B @ theta
    y = mu + rng.normal(0.0, cfg['sigma'], size=mu.shape[0])
    true_params = {
        'beta': beta.tolist(), 'sigma': cfg['sigma'], 'sigma_theta': cfg['sigma_theta'],
        'phi_s': cfg['phi_s'], 'phi_t': cfg['phi_t'], 'nu_s': cfg['nu_s'], 'nu_t': cfg['nu_t'],
        'lambda': cfg['lambda']
    }
    return dict(y=y, X=X, B=B, ST=ST, S=S, T=T, centers_s=centers_s,
                theta=theta, beta=beta, true_parameters=true_params, basis_scale=basis_scale)


In [ ]:

# =====================================================
# Variogram fitting functions from finalized Notebook 2/3
# =====================================================
def semivariogram_model(d, sill, phi, nu, nugget):
    return nugget + sill * (1.0 - matern_corr(d, phi, nu))

def bin_variogram(dist, gamma_vals, n_bins=12, min_pairs=20):
    dist = np.asarray(dist)
    gamma_vals = np.asarray(gamma_vals)
    mask = np.isfinite(dist) & np.isfinite(gamma_vals) & (dist > 0)
    dist = dist[mask]
    gamma_vals = gamma_vals[mask]
    edges = np.unique(np.quantile(dist, np.linspace(0, 1, n_bins + 1)))
    centers, gammas, counts = [], [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (dist >= a) & (dist <= b)
        if m.sum() >= min_pairs:
            centers.append(float(np.mean(dist[m])))
            gammas.append(float(np.mean(gamma_vals[m])))
            counts.append(int(m.sum()))
    return np.asarray(centers), np.asarray(gammas), np.asarray(counts)

def fit_matern_variogram_regularized(d, g, counts, phi_bounds, nu_bounds, prior_phi, prior_nu, prior_weight):
    d = np.asarray(d, dtype='float64')
    g = np.asarray(g, dtype='float64')
    counts = np.asarray(counts, dtype='float64')
    mask = np.isfinite(d) & np.isfinite(g) & (d > 0) & (g >= 0)
    d, g, counts = d[mask], g[mask], counts[mask]
    if len(d) < 4:
        return dict(sill=float(np.nanvar(g) if len(g) else 1.0), phi=float(prior_phi), nu=float(prior_nu), nugget=0.0, se_phi=np.nan, se_nu=np.nan)

    sill0 = max(float(np.nanpercentile(g, 85)), 1e-4)
    nugget0 = max(float(np.nanmin(g)), 0.0)
    p0 = [sill0, float(prior_phi), float(prior_nu), nugget0]
    lower = [1e-6, phi_bounds[0], nu_bounds[0], 0.0]
    upper = [max(10.0 * sill0, 1.0), phi_bounds[1], nu_bounds[1], max(2.0 * sill0, 1.0)]
    sigma_w = 1.0 / np.sqrt(np.maximum(counts, 1.0))

    def augmented_model(d_aug, sill, phi, nu, nugget):
        # last two pseudo-observations regularize phi and nu toward scenario-level values
        d_main = d_aug[:-2]
        return np.r_[semivariogram_model(d_main, sill, phi, nu, nugget), phi, nu]

    d_aug = np.r_[d, 0.0, 0.0]
    g_aug = np.r_[g, prior_phi, prior_nu]
    sigma_aug = np.r_[sigma_w, 1.0 / max(prior_weight, 1e-6), 1.0 / max(prior_weight, 1e-6)]

    try:
        popt, pcov = curve_fit(
            augmented_model, d_aug, g_aug, p0=p0, bounds=(lower, upper),
            sigma=sigma_aug, absolute_sigma=False, maxfev=20000
        )
        se = np.sqrt(np.maximum(np.diag(pcov), 0.0)) if pcov is not None else np.repeat(np.nan, 4)
    except Exception:
        popt = np.asarray(p0, dtype='float64')
        se = np.repeat(np.nan, 4)
    sill, phi, nu, nugget = popt
    return dict(sill=float(sill), phi=float(phi), nu=float(nu), nugget=float(nugget), se_phi=float(se[1]), se_nu=float(se[2]))

def ci95(est, se, lower, upper):
    if not np.isfinite(se) or se <= 0:
        width = 0.25 * max(abs(est), 1.0)
        return float(max(lower, est - width)), float(min(upper, est + width))
    return float(max(lower, est - 1.96 * se)), float(min(upper, est + 1.96 * se))

def estimate_matern_by_variogram(y, X, ST, centers_s, cfg, rng):
    # OLS residuals for fast structural estimation
    beta_ols = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta_ols
    st_x, st_y, st_t = ST[:, 0], ST[:, 1], ST[:, 2]
    unique_t = np.unique(st_t)

    # Spatial variogram: same time pairs
    pairs_d_s, pairs_g_s = [], []
    for tt in unique_t:
        idx = np.where(st_t == tt)[0]
        coords_tt = np.column_stack([st_x[idx], st_y[idx]])
        vals = resid[idx]
        pairs_d_s.append(pdist(coords_tt))
        pairs_g_s.append(0.5 * pdist(vals[:, None], metric='sqeuclidean'))
    pairs_d_s = np.concatenate(pairs_d_s)
    pairs_g_s = np.concatenate(pairs_g_s)
    if len(pairs_d_s) > cfg['max_pairs_space']:
        ii = rng.choice(len(pairs_d_s), size=cfg['max_pairs_space'], replace=False)
        pairs_d_s, pairs_g_s = pairs_d_s[ii], pairs_g_s[ii]
    d_s, g_s, c_s = bin_variogram(pairs_d_s, pairs_g_s, cfg['n_bins_space'], cfg['min_pairs_per_bin'])
    fit_s = fit_matern_variogram_regularized(
        d_s, g_s, c_s, cfg['phi_s_bounds'], cfg['nu_s_bounds'],
        cfg['prior_phi_s'], cfg['prior_nu_s'], cfg['variogram_prior_weight']
    )

    # Temporal variogram: same spatial site pairs
    pairs_d_t, pairs_g_t = [], []
    site_key = pd.Series(list(zip(np.round(st_x, 8), np.round(st_y, 8))))
    site_df = pd.DataFrame({'site': site_key, 'time': st_t, 'resid': resid})
    for _, grp in site_df.groupby('site'):
        if len(grp) < 2:
            continue
        times = grp['time'].to_numpy(dtype='float64')
        vals = grp['resid'].to_numpy(dtype='float64')
        pairs_d_t.append(pdist(times[:, None]))
        pairs_g_t.append(0.5 * pdist(vals[:, None], metric='sqeuclidean'))
    pairs_d_t = np.concatenate(pairs_d_t)
    pairs_g_t = np.concatenate(pairs_g_t)
    if len(pairs_d_t) > cfg['max_pairs_time']:
        ii = rng.choice(len(pairs_d_t), size=cfg['max_pairs_time'], replace=False)
        pairs_d_t, pairs_g_t = pairs_d_t[ii], pairs_g_t[ii]
    d_t, g_t, c_t = bin_variogram(pairs_d_t, pairs_g_t, cfg['n_bins_time'], cfg['min_pairs_per_bin'])
    fit_t = fit_matern_variogram_regularized(
        d_t, g_t, c_t, cfg['phi_t_bounds'], cfg['nu_t_bounds'],
        cfg['prior_phi_t'], cfg['prior_nu_t'], cfg['variogram_prior_weight']
    )

    phi_s_hat, nu_s_hat = fit_s['phi'], fit_s['nu']
    phi_t_hat, nu_t_hat = fit_t['phi'], fit_t['nu']
    phi_s_lo, phi_s_hi = ci95(phi_s_hat, fit_s['se_phi'], *cfg['phi_s_bounds'])
    nu_s_lo, nu_s_hi = ci95(nu_s_hat, fit_s['se_nu'], *cfg['nu_s_bounds'])
    phi_t_lo, phi_t_hi = ci95(phi_t_hat, fit_t['se_phi'], *cfg['phi_t_bounds'])
    nu_t_lo, nu_t_hi = ci95(nu_t_hat, fit_t['se_nu'], *cfg['nu_t_bounds'])

    return {
        'phi_s_hat': phi_s_hat, 'phi_t_hat': phi_t_hat, 'nu_s_hat': nu_s_hat, 'nu_t_hat': nu_t_hat,
        'phi_s_lo': phi_s_lo, 'phi_s_hi': phi_s_hi, 'phi_t_lo': phi_t_lo, 'phi_t_hi': phi_t_hi,
        'nu_s_lo': nu_s_lo, 'nu_s_hi': nu_s_hi, 'nu_t_lo': nu_t_lo, 'nu_t_hi': nu_t_hi,
        'fit_s': fit_s, 'fit_t': fit_t
    }


In [ ]:

# =====================================================
# PyMC fit functions from finalized Notebook 2/3
# =====================================================
def build_L_theta_hat(centers_s, K, J, r, matern_est, cfg):
    D_centers = squareform(pdist(centers_s))
    Cs_hat = matern_corr(D_centers, matern_est['phi_s_hat'], matern_est['nu_s_hat']) + cfg['jitter'] * np.eye(K)
    D_basis = squareform(pdist(np.arange(J, dtype='float64')[:, None]))
    Ct_hat = matern_corr(D_basis, matern_est['phi_t_hat'], matern_est['nu_t_hat']) + cfg['jitter'] * np.eye(J)
    C_theta_hat = np.kron(Ct_hat, Cs_hat) + cfg['jitter'] * np.eye(r)
    jitter = cfg['jitter']
    for _ in range(8):
        try:
            return np.linalg.cholesky(C_theta_hat + jitter * np.eye(r)).astype('float64')
        except np.linalg.LinAlgError:
            jitter *= 10
    raise np.linalg.LinAlgError('Could not compute Cholesky for C_theta_hat')

def fit_pymc_method(y, X, B, L_theta_hat, true_params, cfg, method, seed):
    N, p = X.shape
    r = B.shape[1]
    true_beta = np.array(true_params['beta'], dtype='float64')
    true_sigma = float(true_params['sigma'])
    true_sigma_theta = float(true_params['sigma_theta'])
    true_lambda = float(true_params['lambda'])

    start = time.time()
    coords = {'obs_id': np.arange(N), 'beta_id': np.arange(p), 'theta_id': np.arange(r)}
    with pm.Model(coords=coords) as model:
        X_data = pm.Data('X', X, dims=('obs_id', 'beta_id'))
        B_data = pm.Data('B_model', B, dims=('obs_id', 'theta_id'))
        L_data = pm.Data('L_theta_hat', L_theta_hat, dims=('theta_id', 'theta_id'))
        beta = pm.Normal('beta', mu=0.0, sigma=cfg['beta_sd'], dims='beta_id')
        sigma = pm.HalfNormal('sigma', sigma=cfg['sigma_halfnormal_sd'])

        sigma_theta_fixed = float(true_sigma_theta)
        lambda_raw = pm.Normal('lambda_raw', mu=0.0, sigma=cfg['lambda_sd'])
        lambda_param = pm.Deterministic('lambda', pm.math.clip(lambda_raw, cfg['lambda_lower'], cfg['lambda_upper']))
        z_theta = pm.Normal('z_theta', mu=0.0, sigma=1.0, dims='theta_id')
        theta = pm.Deterministic('theta', sigma_theta_fixed * pm.math.dot(L_data, z_theta), dims='theta_id')
        mu = pm.math.dot(X_data, beta) + pm.math.dot(B_data, theta)
        pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y, dims='obs_id')

        initvals = {
            'beta': np.linalg.lstsq(X, y, rcond=None)[0],
            'sigma_log__': np.log(max(np.std(y - X @ np.linalg.lstsq(X, y, rcond=None)[0]), 0.5)),
            'lambda_raw': 0.0,
            'z_theta': np.zeros(r)
        }
        if method == 'AHMC_NUTS':
            idata = pm.sample(
                draws=cfg['nuts_draws'], tune=cfg['nuts_tune'], chains=cfg['nuts_chains'], cores=cfg['nuts_cores'],
                target_accept=cfg['nuts_target_accept'], nuts_sampler_kwargs={'max_treedepth': cfg['nuts_max_treedepth']},
                random_seed=[seed + 10, seed + 20][:cfg['nuts_chains']], initvals=initvals,
                progressbar=False, compute_convergence_checks=False, return_inferencedata=True
            )
        elif method == 'MCMC_Metropolis':
            step = pm.Metropolis()
            idata = pm.sample(
                draws=cfg['metro_draws'], tune=cfg['metro_tune'], chains=cfg['metro_chains'], cores=cfg['metro_cores'],
                step=step, random_seed=[seed + 30, seed + 40][:cfg['metro_chains']], initvals=initvals,
                progressbar=False, compute_convergence_checks=False, return_inferencedata=True
            )
        else:
            raise ValueError('Unknown method: ' + str(method))

    runtime_seconds = time.time() - start

    def stat(var_name, index=None, true_value=None, label=None):
        arr = idata.posterior[var_name].values
        if index is not None:
            arr = arr[..., index]
        vals = arr.reshape(-1)
        est = float(vals.mean())
        lo, hi = np.quantile(vals, [0.025, 0.975])
        return {
            'parameter': label or var_name, 'true': float(true_value), 'estimate': est,
            'squared_error': float((est - true_value) ** 2),
            'lower_95': float(lo), 'upper_95': float(hi),
            'covered_95': int(lo <= true_value <= hi), 'method': method,
            'runtime_seconds': float(runtime_seconds)
        }

    rows = [stat('beta', index=j, true_value=true_beta[j], label=f'beta{j}') for j in range(p)]
    rows.append(stat('sigma', true_value=true_sigma, label='sigma'))
    rows.append({
        'parameter': 'sigma_theta', 'true': true_sigma_theta, 'estimate': true_sigma_theta,
        'squared_error': 0.0, 'lower_95': true_sigma_theta, 'upper_95': true_sigma_theta,
        'covered_95': 1, 'method': 'Fixed_sigma_theta', 'runtime_seconds': float(runtime_seconds)
    })
    rows.append(stat('lambda', true_value=true_lambda, label='lambda'))
    return pd.DataFrame(rows), idata, runtime_seconds

def matern_metrics_df(matern_est, true_params, runtime_seconds):
    true_phi_s = float(true_params['phi_s']); true_phi_t = float(true_params['phi_t'])
    true_nu_s = float(true_params['nu_s']); true_nu_t = float(true_params['nu_t'])
    rows = [
        ('phi_s', true_phi_s, matern_est['phi_s_hat'], matern_est['phi_s_lo'], matern_est['phi_s_hi']),
        ('phi_t', true_phi_t, matern_est['phi_t_hat'], matern_est['phi_t_lo'], matern_est['phi_t_hi']),
        ('nu_s', true_nu_s, matern_est['nu_s_hat'], matern_est['nu_s_lo'], matern_est['nu_s_hi']),
        ('nu_t', true_nu_t, matern_est['nu_t_hat'], matern_est['nu_t_lo'], matern_est['nu_t_hi']),
    ]
    return pd.DataFrame([{
        'parameter': name, 'true': tv, 'estimate': est, 'squared_error': float((est - tv) ** 2),
        'lower_95': lo, 'upper_95': hi, 'covered_95': int(lo <= tv <= hi),
        'method': 'Empirical_variogram_Matern', 'runtime_seconds': float(runtime_seconds)
    } for name, tv, est, lo, hi in rows])


In [ ]:

# =====================================================
# Main simulation loop
# =====================================================
all_rows = []
failed_reps = []

for rep in range(1, CONFIG['N_REPS'] + 1):
    rep_seed = CONFIG['base_seed'] + 1000 * rep
    rep_rng = np.random.default_rng(rep_seed)
    print('\n' + '=' * 70)
    print(f'Replication {rep}/{CONFIG["N_REPS"]} | seed={rep_seed}')
    print('=' * 70)

    try:
        # Generate fresh dataset
        data = simulate_lowrank_dataset(CONFIG, rep_seed)
        y, X, B, ST = data['y'], data['X'], data['B'], data['ST']
        centers_s = data['centers_s']
        true_params = data['true_parameters']
        K, J, r = CONFIG['K'], CONFIG['J'], CONFIG['K'] * CONFIG['J']

        # Estimate structural Matern parameters by empirical variogram, same as final Notebook 2/3
        matern_est = estimate_matern_by_variogram(y, X, ST, centers_s, CONFIG, rep_rng)
        print('Matern estimates:', {k: round(v, 4) for k, v in matern_est.items() if k.endswith('_hat')})

        # Cholesky for theta covariance from estimated Matern parameters
        L_theta_hat = build_L_theta_hat(centers_s, K, J, r, matern_est, CONFIG)

        # AHMC = NUTS
        print('Running AHMC/NUTS ...')
        ahmc_df, ahmc_idata, ahmc_time = fit_pymc_method(
            y, X, B, L_theta_hat, true_params, CONFIG, method='AHMC_NUTS', seed=rep_seed
        )
        ahmc_matern_df = matern_metrics_df(matern_est, true_params, ahmc_time)
        ahmc_full = pd.concat([ahmc_df, ahmc_matern_df], ignore_index=True)
        ahmc_full['replication'] = rep
        all_rows.append(ahmc_full)
        del ahmc_idata
        gc.collect()

        # Metropolis
        print('Running MCMC/Metropolis ...')
        metro_df, metro_idata, metro_time = fit_pymc_method(
            y, X, B, L_theta_hat, true_params, CONFIG, method='MCMC_Metropolis', seed=rep_seed + 500
        )
        metro_matern_df = matern_metrics_df(matern_est, true_params, metro_time)
        metro_full = pd.concat([metro_df, metro_matern_df], ignore_index=True)
        metro_full['replication'] = rep
        all_rows.append(metro_full)
        del metro_idata
        gc.collect()

        # Save incrementally after every replication
        raw_so_far = pd.concat(all_rows, ignore_index=True)
        raw_so_far.to_csv(os.path.join(CONFIG['results_dir'], 'simulation_raw_results_lambda0.csv'), index=False)
        print('Saved raw results so far:', raw_so_far.shape)

    except Exception as e:
        print('FAILED replication', rep, repr(e))
        failed_reps.append({'replication': rep, 'error': repr(e)})
        pd.DataFrame(failed_reps).to_csv(os.path.join(CONFIG['results_dir'], 'failed_reps.csv'), index=False)
        gc.collect()

raw_results = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
print('\nFinished. Raw shape:', raw_results.shape)
raw_results.head()


In [ ]:

# =====================================================
# Summarize MSE, CP, and runtime for Table 1, scenario 1
# =====================================================
if raw_results.empty:
    raise RuntimeError('No successful simulation results were produced.')

# Do not report sigma_theta in final paper table if not needed, but keep it in raw file.
summary = (
    raw_results
    .groupby(['parameter', 'method'], as_index=False)
    .agg(
        MSE=('squared_error', 'mean'),
        CP_percent=('covered_95', lambda x: 100.0 * np.mean(x)),
        mean_estimate=('estimate', 'mean'),
        sd_estimate=('estimate', 'std'),
        mean_runtime_seconds=('runtime_seconds', 'mean'),
        n_success=('replication', 'nunique')
    )
)

# A cleaner table for the exact paper rows: AHMC vs MCMC only.
def method_for_table(row):
    if row['method'] == 'AHMC_NUTS': return 'AHMC'
    if row['method'] == 'MCMC_Metropolis': return 'MCMC'
    if row['method'] == 'Empirical_variogram_Matern': return 'Structural'
    if row['method'] == 'Fixed_sigma_theta': return 'Fixed'
    return row['method']
summary['method_table'] = summary.apply(method_for_table, axis=1)

raw_path = os.path.join(
    CONFIG["results_dir"],
    f"simulation_raw_results_lambda0_part{PART:02d}.csv"
)

summary_path = os.path.join(
    CONFIG["results_dir"],
    f"simulation_summary_lambda0_part{PART:02d}.csv"
)

raw_results.to_csv(raw_path, index=False)
summary.to_csv(summary_path, index=False)

with open(os.path.join(CONFIG['results_dir'], 'config_notebook4.json'), 'w', encoding='utf-8') as f:
    json.dump(CONFIG, f, indent=4, ensure_ascii=False)

print('Saved:', raw_path)
print('Saved:', summary_path)
summary


In [ ]:

# =====================================================
# Display paper-oriented scenario-1 summary
# For beta/sigma/lambda: use AHMC_NUTS and MCMC_Metropolis.
# For phi/nu: the same structural estimate is used for both methods in the paper table.
# =====================================================
params_order = ['beta0','beta1','beta2','sigma','phi_s','phi_t','nu_s','nu_t','lambda']

paper_rows = []
for par in params_order:
    if par in ['phi_s','phi_t','nu_s','nu_t']:
        s = summary[(summary['parameter'] == par) & (summary['method'] == 'Empirical_variogram_Matern')]
        if len(s):
            for meth in ['AHMC','MCMC']:
                paper_rows.append({
                    'parameter': par, 'method': meth,
                    'MSE_case1': float(s['MSE'].iloc[0]),
                    'CP_case1_percent': float(s['CP_percent'].iloc[0]),
                    'mean_runtime_seconds': float(s['mean_runtime_seconds'].iloc[0])
                })
    else:
        for source_method, meth in [('AHMC_NUTS','AHMC'), ('MCMC_Metropolis','MCMC')]:
            s = summary[(summary['parameter'] == par) & (summary['method'] == source_method)]
            if len(s):
                paper_rows.append({
                    'parameter': par, 'method': meth,
                    'MSE_case1': float(s['MSE'].iloc[0]),
                    'CP_case1_percent': float(s['CP_percent'].iloc[0]),
                    'mean_runtime_seconds': float(s['mean_runtime_seconds'].iloc[0])
                })

paper_summary = pd.DataFrame(paper_rows)
paper_summary.to_csv(os.path.join(CONFIG['results_dir'], 'paper_table_case1_ready.csv'), index=False)
paper_summary


In [ ]:

# =====================================================
# Optional: clean memory before Kaggle Save Version
# =====================================================
try:
    del raw_results, all_rows
except Exception:
    pass
gc.collect()
print('Notebook 4 completed. Files in results_dir:')
for fn in sorted(os.listdir(CONFIG['results_dir'])):
    fp = os.path.join(CONFIG['results_dir'], fn)
    print(f'{fn:45s} {os.path.getsize(fp)/1024:.1f} KB')
